# CUDA Lab 01



In [ ]:
!nvidia-smi
import subprocess, re
RESULTS = {}
_o = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
RESULTS["driver"] = re.search(r"Driver Version:\s*([\d.]+)", _o).group(1)
RESULTS["cuda_driver"] = re.search(r"CUDA Version:\s*([\d.]+)", _o).group(1)

In [ ]:
from numba import cuda
import os
device = cuda.get_current_device()
_name = device.name.decode("utf-8") if isinstance(device.name, bytes) else device.name
print(f"Device Name: {_name}")
print(f"Compute Capability: {device.compute_capability}")
print(f"Multiprocessor Count (SMs): {getattr(device, 'MULTIPROCESSOR_COUNT', 'N/A')}")
print(f"Max Threads Per Block: {device.MAX_THREADS_PER_BLOCK}")
print(f"Max Block Dimensions: {device.MAX_BLOCK_DIM_X}, {device.MAX_BLOCK_DIM_Y}, {device.MAX_BLOCK_DIM_Z}")
print(f"Warp Size: {device.WARP_SIZE}")
free_b, total_b = cuda.current_context().get_memory_info()
print(f"Total VRAM: {total_b/1024**2:.0f} MiB")
RESULTS.update(gpu=_name, cc=f"{device.compute_capability[0]}.{device.compute_capability[1]}",
               max_threads=device.MAX_THREADS_PER_BLOCK, vram=f"{total_b/1024**2:.0f} MiB")

## Section 2: Crossover experiment

In [ ]:
import numpy as np
import time
from numba import cuda

def double_cpu_pure(arr):
    out = np.empty_like(arr)
    for i in range(len(arr)):
        out[i] = arr[i] * 2.0
    return out

@cuda.jit
def double_gpu_kernel(d_in, d_out):
    idx = cuda.grid(1)
    if idx < d_in.shape[0]:
        d_out[idx] = d_in[idx] * 2.0

In [ ]:
def benchmark_run(N, student_id_seed=1234):
    np.random.seed(student_id_seed)
    h_in = np.random.rand(N).astype(np.float32)
    if N <= 1_000_000:
        t0 = time.perf_counter()
        _ = double_cpu_pure(h_in)
        cpu_time_ms = (time.perf_counter() - t0) * 1000.0
    else:
        cpu_time_ms = float('nan')
    threads_per_block = 256
    blocks_per_grid = (N + threads_per_block - 1) // threads_per_block
    d_warmup = cuda.to_device(np.ones(10, dtype=np.float32))
    double_gpu_kernel[1, 32](d_warmup, d_warmup)
    cuda.synchronize()
    t0 = time.perf_counter()
    d_in = cuda.to_device(h_in)
    d_out = cuda.device_array_like(d_in)
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    h_out = d_out.copy_to_host()
    cuda.synchronize()
    gpu_total_ms = (time.perf_counter() - t0) * 1000.0
    t0 = time.perf_counter()
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    cuda.synchronize()
    gpu_compute_ms = (time.perf_counter() - t0) * 1000.0
    return cpu_time_ms, gpu_total_ms, gpu_compute_ms

sizes = [100, 1_000, 10_000, 100_000, 1_000_000, 5_000_000, 10_000_000]
rows = []
print(f"{'N':>12} {'CPU ms':>12} {'GPU total':>12} {'GPU kernel':>12}  Winner")
for N in sizes:
    cpu, tot, ker = benchmark_run(N)
    if cpu != cpu:
        winner = "GPU (CPU skipped)"
    else:
        winner = "GPU" if tot < cpu else "CPU"
    rows.append((N, cpu, tot, ker, winner))
    print(f"{N:>12,} {cpu:>12.4f} {tot:>12.4f} {ker:>12.4f}  {winner}")

crossover = next((r[0] for r in rows if r[1] == r[1] and r[2] < r[1]), None)
CROSSOVER_N = crossover
n100 = rows[0]
overhead_pct = (n100[2] - n100[3]) / n100[2] * 100
print(f"\nCrossover N* = {CROSSOVER_N}")
print(f"At N=100: GPU total {n100[2]:.4f} ms vs kernel only {n100[3]:.4f} ms -> {overhead_pct:.1f}% of GPU total is NOT kernel compute (transfer/launch overhead)")
RESULTS["table"] = rows; RESULTS["crossover"] = CROSSOVER_N; RESULTS["overhead_pct"] = overhead_pct

## Section 3.1: Missing guard
If this cell raises CUDA_ERROR_ILLEGAL_ADDRESS, the context is dead: copy the error text, then Runtime -> Restart session and re-run the other cells (skip this one).

In [ ]:
@cuda.jit
def defective_kernel(d_arr):
    idx = cuda.grid(1)
    d_arr[idx] = 999.0

N = 1000
h_arr = np.zeros(N, dtype=np.float32)
d_arr = cuda.to_device(h_arr)
threads_per_block = 256
blocks_per_grid = 4
print("Launching defective kernel with 1024 threads on 1000-element array...")
defective_kernel[blocks_per_grid, threads_per_block](d_arr)
cuda.synchronize()
print("Execution finished (or did it crash?). Checking host copy...")
h_result = d_arr.copy_to_host()
print(f"Elements processed: {len(h_result)}. Last element: {h_result[-1]}")

## Section 3.2: Thread coordinates (N = 37)

In [ ]:
@cuda.jit
def inspect_threads_kernel(log_block, log_thread, log_global, log_active, N):
    t_idx = cuda.threadIdx.x
    b_idx = cuda.blockIdx.x
    g_idx = cuda.grid(1)
    if g_idx < log_block.shape[0]:
        log_block[g_idx] = b_idx
        log_thread[g_idx] = t_idx
        log_global[g_idx] = g_idx
        log_active[g_idx] = 1 if g_idx < N else 0

N = 37
threads_per_block = 8
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block
total_threads = blocks_per_grid * threads_per_block
log_b = cuda.device_array(total_threads, dtype=np.int32)
log_t = cuda.device_array(total_threads, dtype=np.int32)
log_g = cuda.device_array(total_threads, dtype=np.int32)
log_a = cuda.device_array(total_threads, dtype=np.int32)
inspect_threads_kernel[blocks_per_grid, threads_per_block](log_b, log_t, log_g, log_a, N)
cuda.synchronize()
lb, lt, lg, la = (x.copy_to_host() for x in (log_b, log_t, log_g, log_a))
print(f"{'Global ID':<10} | {'Block ID':<10} | {'Thread ID':<10} | {'Status':<10}")
print("-" * 50)
for i in range(32, 40):
    status = "ACTIVE" if la[i] == 1 else "IDLE (GUARDED)"
    print(f"{lg[i]:<10} | {lb[i]:<10} | {lt[i]:<10} | {status:<10}")
active = int(la.sum())
print(f"\nBlocks: {blocks_per_grid} | Threads launched: {total_threads} | Active: {active} | Idle: {total_threads-active}")
RESULTS["geom37"] = (blocks_per_grid, total_threads, active, total_threads-active)

## Section 4: Race condition

In [ ]:
@cuda.jit
def naive_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
        d_total[0] += d_arr[idx]

@cuda.jit
def atomic_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
        cuda.atomic.add(d_total, 0, d_arr[idx])

M = 50_000
d_ones = cuda.to_device(np.ones(M, dtype=np.float32))
tpb = 256
bpg = (M + tpb - 1) // tpb

# warmup compile (not timed)
_w = cuda.to_device(np.zeros(1, dtype=np.float32))
naive_sum_kernel[bpg, tpb](d_ones, _w); atomic_sum_kernel[bpg, tpb](d_ones, _w); cuda.synchronize()

naive_trials = []
print(f"{'Trial':<8}{'Target':>10}{'Naive output':>16}{'Deviation':>14}")
for t in range(1, 6):
    d_total = cuda.to_device(np.zeros(1, dtype=np.float32))
    naive_sum_kernel[bpg, tpb](d_ones, d_total)
    cuda.synchronize()
    out = float(d_total.copy_to_host()[0])
    naive_trials.append(out)
    print(f"{t:<8}{50000.0:>10.1f}{out:>16.1f}{50000.0-out:>14.1f}")

d_total = cuda.to_device(np.zeros(1, dtype=np.float32))
atomic_sum_kernel[bpg, tpb](d_ones, d_total); cuda.synchronize()
atomic_out = float(d_total.copy_to_host()[0])
print(f"\nAtomic sum output: {atomic_out}  (exactly 50000.0? {'Yes' if atomic_out == 50000.0 else 'No'})")

def time_kernel(k, reps=20):
    ts = []
    for _ in range(reps):
        d_t = cuda.to_device(np.zeros(1, dtype=np.float32))
        cuda.synchronize()
        t0 = time.perf_counter()
        k[bpg, tpb](d_ones, d_t)
        cuda.synchronize()
        ts.append((time.perf_counter() - t0) * 1000.0)
    return float(np.median(ts))
naive_ms = time_kernel(naive_sum_kernel)
atomic_ms = time_kernel(atomic_sum_kernel)
print(f"Naive kernel: {naive_ms:.4f} ms | Atomic kernel: {atomic_ms:.4f} ms (median of 20)")
RESULTS.update(naive_trials=naive_trials, atomic_out=atomic_out, naive_ms=naive_ms, atomic_ms=atomic_ms)

## Section 5: 2D vignette

In [ ]:
import math

@cuda.jit
def radial_vignette_2d(d_canvas, width, height, center_x, center_y):
    x, y = cuda.grid(2)
    if x < width and y < height:
        dx = float(x - center_x)
        dy = float(y - center_y)
        dist = math.sqrt(dx * dx + dy * dy)
        max_dist = math.sqrt(float(center_x**2 + center_y**2))
        intensity = 1.0 - (dist / max_dist)
        if intensity < 0.0:
            intensity = 0.0
        d_canvas[y, x] = intensity

WIDTH, HEIGHT = 1024, 1024
canvas = np.zeros((HEIGHT, WIDTH), dtype=np.float32)
d_canvas = cuda.to_device(canvas)
threads_2d = (16, 16)
blocks_2d = ((WIDTH + threads_2d[0] - 1) // threads_2d[0],
             (HEIGHT + threads_2d[1] - 1) // threads_2d[1])
radial_vignette_2d[blocks_2d, threads_2d](d_canvas, WIDTH, HEIGHT, WIDTH // 2, HEIGHT // 2)
cuda.synchronize()
result_canvas = d_canvas.copy_to_host()
print(f"Generated 2D matrix shape: {result_canvas.shape}. Center value: {result_canvas[512, 512]:.3f}")
print(f"Blocks X: {blocks_2d[0]} | Blocks Y: {blocks_2d[1]} | Total: {blocks_2d[0]*blocks_2d[1]}")
print(f"Center (512,512): {result_canvas[512,512]:.6f} | Corner (0,0): {result_canvas[0,0]:.6f}")
RESULTS["grid2d"] = (blocks_2d[0], blocks_2d[1], blocks_2d[0]*blocks_2d[1])
RESULTS["center"] = float(result_canvas[512,512]); RESULTS["corner"] = float(result_canvas[0,0])

## Section 6: Checksum

In [ ]:
import hashlib
STUDENT_ID = "230103342"

def generate_lab_checksum(student_id_str, d_canvas, exp_table_crossover):
    h = hashlib.sha256()
    h.update(student_id_str.strip().encode('utf-8'))
    nm = cuda.get_current_device().name
    h.update(nm if isinstance(nm, bytes) else nm.encode('utf-8'))
    h.update(d_canvas.copy_to_host()[:10, :10].tobytes())
    h.update(str(exp_table_crossover).encode('utf-8'))
    digest = h.hexdigest()[:16].upper()
    print("=" * 60)
    print(f"OFFICIAL VERIFICATION CHECKSUM TOKEN: {digest}")
    print("=" * 60)
    return digest

TOKEN = generate_lab_checksum(STUDENT_ID, d_canvas, CROSSOVER_N)

## Summary to copy into the task sheet

In [ ]:
R = RESULTS
print("Driver:", R["driver"], "| CUDA:", R["cuda_driver"], "| GPU:", R["gpu"], "| VRAM:", R["vram"], "| CC:", R["cc"], "| Max threads/block:", R["max_threads"])
for r in R["table"]:
    print(f"N={r[0]:>10,} CPU={r[1]:.4f} GPUtot={r[2]:.4f} GPUker={r[3]:.4f} {r[4]}")
print("Crossover N*:", R["crossover"], "| N=100 overhead %:", round(R["overhead_pct"],1))
print("37/8 -> blocks, threads, active, idle:", R["geom37"])
print("Naive trials:", R["naive_trials"], "| Atomic:", R["atomic_out"], "| naive ms:", round(R["naive_ms"],4), "| atomic ms:", round(R["atomic_ms"],4))
print("2D grid:", R["grid2d"], "| center:", R["center"], "| corner:", R["corner"])
print("TOKEN:", TOKEN)